In [ ]:
# DNN Assignment Phase 1 — ANN Model
# Alzheimer's Disease Classification (4-class)
# Dataset: Augmented Alzheimer MRI Dataset (Kaggle/uraninjo)


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


config = {
    "lr": 0.002,
    "epochs": 20,
    "batch_size": 32,
    "dropout": 0.5,
    "hidden_layers": [512, 256, 128],
    "optimizer": "adam",
    "weight_decay": 1e-4,
    "img_size": 128,
}


TRAIN_DIR = "/kaggle/input/datasets/uraninjo/augmented-alzheimer-mri-dataset/AugmentedAlzheimerDataset"
VAL_DIR   = "/kaggle/input/datasets/uraninjo/augmented-alzheimer-mri-dataset/OriginalDataset"
SAVE_PATH = "/kaggle/working/ann_checkpoint.pth"

CLASS_NAMES = ["MildDemented", "ModerateDemented", "NonDemented", "VeryMildDemented"]
NUM_CLASSES = 4
IMG_SIZE    = config["img_size"]
INPUT_SIZE  = IMG_SIZE * IMG_SIZE  # 16384


train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])


train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset   = datasets.ImageFolder(VAL_DIR,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=config["batch_size"], shuffle=False, num_workers=2)

print(f"Train samples : {len(train_dataset)}")
print(f"Val samples   : {len(val_dataset)}")
print(f"Classes       : {train_dataset.classes}")


class ANN(nn.Module):
    def __init__(self, input_size, hidden_layers, num_classes, dropout):
        super(ANN, self).__init__()
        layers = []
        in_size = input_size
        for h in hidden_layers:
            layers += [
                nn.Linear(in_size, h),
                nn.ReLU(),
                nn.Dropout(p=dropout),
            ]
            in_size = h
        layers.append(nn.Linear(in_size, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)   # flatten: [B, 1, 128, 128] → [B, 16384]
        return self.network(x)       # logits; CrossEntropyLoss handles softmax

model = ANN(INPUT_SIZE, config["hidden_layers"], NUM_CLASSES, config["dropout"]).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])


train_acc_history = []
val_acc_history   = []
train_loss_history = []
val_loss_history   = []

best_val_acc = 0.0

for epoch in range(config["epochs"]):
    # ── Train ──
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total


    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total   = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total   += labels.size(0)

    val_loss = val_loss / val_total
    val_acc  = val_correct / val_total

    train_acc_history.append(train_acc)
    val_acc_history.append(val_acc)
    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)

    print(f"Epoch [{epoch+1:02d}/{config['epochs']}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

    # Save best checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch"      : epoch + 1,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_acc"    : val_acc,
            "config"     : config,
        }, SAVE_PATH)
        print(f"  → Checkpoint saved (best val acc: {best_val_acc*100:.2f}%)")

print(f"\nTraining complete. Best val accuracy: {best_val_acc*100:.2f}%")


model.eval()
all_preds  = []
all_labels = []
all_probs  = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1)
        _, preds = outputs.max(1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.numpy())
        all_probs.append(probs.cpu().numpy())

all_preds  = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)
all_probs  = np.concatenate(all_probs)


epochs_range = range(1, config["epochs"] + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs_range, [a * 100 for a in train_acc_history], marker='o', label='Train Accuracy', color='steelblue')
plt.plot(epochs_range, [a * 100 for a in val_acc_history],   marker='s', label='Val Accuracy',   color='darkorange')
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("ANN — Training vs Validation Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/ann_accuracy_curve.png", dpi=150)
plt.show()
print("Saved: ann_accuracy_curve.png")


plt.figure(figsize=(8, 5))
plt.plot(epochs_range, train_loss_history, marker='o', label='Train Loss', color='steelblue')
plt.plot(epochs_range, val_loss_history,   marker='s', label='Val Loss',   color='darkorange')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ANN — Training vs Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/ann_loss_curve.png", dpi=150)
plt.show()
print("Saved: ann_loss_curve.png")


cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("ANN — Confusion Matrix")
plt.tight_layout()
plt.savefig("/kaggle/working/ann_confusion_matrix.png", dpi=150)
plt.show()
print("Saved: ann_confusion_matrix.png")


from sklearn.metrics import roc_curve, auc

y_bin = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
colors = ['steelblue', 'darkorange', 'green', 'red']

plt.figure(figsize=(8, 6))
for i, (cls, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f"{cls} (AUC = {roc_auc:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ANN — ROC-AUC Curve (4 Classes)")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/ann_roc_auc.png", dpi=150)
plt.show()
print("Saved: ann_roc_auc.png")


print("\n── Final Results ────────────────────────────────")
print(f"Best Validation Accuracy : {best_val_acc*100:.2f}%")
print(f"Final Train Accuracy     : {train_acc_history[-1]*100:.2f}%")
print(f"Final Val Accuracy       : {val_acc_history[-1]*100:.2f}%")

macro_auc = roc_auc_score(y_bin, all_probs, average='macro')
print(f"Macro ROC-AUC            : {macro_auc:.4f}")
print("\nAll 4 KPI plots saved to /kaggle/working/")
print("Checkpoint saved to:", SAVE_PATH)





Using device: cuda
Train samples : 33984
Val samples   : 6400
Classes       : ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
ANN(
  (network): Sequential(
    (0): Linear(in_features=16384, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.5, inplace=False)
    (6): Linear(in_features=256, out_features=128, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.5, inplace=False)
    (9): Linear(in_features=128, out_features=4, bias=True)
  )
)
Trainable parameters: 8,553,860
Epoch [01/20] Train Loss: 1.3664 | Train Acc: 32.52% | Val Loss: 1.1700 | Val Acc: 49.41%
  → Checkpoint saved (best val acc: 49.41%)
Epoch [02/20] Train Loss: 1.3254 | Train Acc: 31.92% | Val Loss: 1.2156 | Val Acc: 29.31%
Epoch [03/20] Train Loss: 1.3183 | Train Acc: 31.71% | Val Loss: 1.1240 | Val Acc: 48.95%
Epoch [04/20] Train Loss: 1.3147 | Train Acc: 31.67% | 